In [ ]:
import numpy as np
import pandas as pd

from os.path import join
from typing import Any
from IPython.display import display, Markdown

In [ ]:
df = pd.read_csv(join('data', 'credit_risk_dataset.csv'))

## 1. Первичный осмотр данных

Перед тем как что-то менять, нужно понять, из чего состоят данные: какие столбцы есть, какого они типа, сколько в них пропусков.

In [ ]:
print('Размер фрейма')
print(df.shape)
print('===========================', end='\n\n')

print('информация о типах')
df.info()
print('===========================', end='\n\n')

print('Пропущенные значения')
missing = df.isnull().sum()
print(missing[missing > 0])
print('===========================', end='\n\n')

print('Распределение целевой переменной')
print(df['loan_status'].value_counts(normalize=True) * 100)
print('===========================', end='\n\n')

display(Markdown('**Описательная статистика**'))
with pd.option_context('display.float_format', '{:.3f}'.format):
    display(df.describe())

In [ ]:
for obj_col in df.select_dtypes('object').columns:
    print(f'Распределение {obj_col}')
    print(df[obj_col].value_counts(normalize=True) * 100)

## 2. Преобразование типов данных

Часть столбцов хранится как обычный текст (`object`), хотя по смыслу это **категориальные переменные** — то есть переменные с ограниченным набором значений. Преобразуем их в тип `category`

Дополнительно:
- `loan_grade` — это **упорядоченная** категория (A лучше, чем B, чем C, и т.д.), поэтому зададим порядок явно;
- `cb_person_default_on_file` — по сути булев признак («да/нет»), переведём Y/N в True/False, это удобнее для модели;
- `loan_status` уже 0/1 — оставляем как есть, но явно понимаем это как целевую переменную (флаг дефолта).

In [ ]:
nominal_cols = ['person_home_ownership', 'loan_intent']
for col in nominal_cols:
    df[col] = df[col].astype('category')


grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
df['loan_grade'] = pd.Categorical(
    df['loan_grade'], 
    categories=grade_order, 
    ordered=True
)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map(
    {'Y': True, 'N': False}
)

df.info()

## 3. Обработка невалидных значений

Есть строки, которые имеют невалидные значения по person_age - значения больше 122,
а также оп person_emp_length - значения больше 122. **Заменим значения на null**

In [ ]:
display(Markdown('**Сортировка по person_age**'))
display(df.sort_values(by='person_age', ascending=False).head(n=6))
display(Markdown('**Сортировка по person_emp_length**'))
display(df.sort_values(by='person_emp_length', ascending=False).head(n=3))

PERSON_AGE_THRESHOLD = 122
PERSON_EMP_LENGTH_THRESHOLD = 122

df.loc[df.person_age > PERSON_AGE_THRESHOLD, 'person_age'] = np.nan
df.loc[df.person_emp_length > PERSON_EMP_LENGTH_THRESHOLD, 'person_emp_length'] = np.nan

## 4. Заполнение пропусков

Прежде чем заполнять пропуски "среднем по больнице", посмотрим, насколько сильно распределение отличается от нормального (есть ли выбросы, асимметрия) — от этого зависит, что лучше использовать: среднее или медиану.

**Правило:** если в столбце есть выбросы или асимметричное распределение, среднее значение будет "перекошено" редкими экстремальными значениями — в таком случае медиана надёжнее.

In [ ]:
for col in ['person_emp_length', 'loan_int_rate']:
    print(f'--- {col} ---')
    print('среднее :', round(df[col].mean(), 2))
    print('медиана :', round(df[col].median(), 2))
    print('станд.откл.:', round(df[col].std(), 2))
    print('пропусков:', df[col].isnull().sum())
    print()

**`loan_int_rate` (процентная ставка).** Ставка по кредиту сильно зависит от кредитного грейда заёмщика (чем хуже грейд — тем выше ставка). Поэтому заполнять пропуски одной общей медианой некорректно: заёмщику с грейдом A и грейдом G медиана "подставит" одинаковую ставку, хотя в реальности они разные.

Правильнее заполнить пропуски **медианой ставки внутри каждого грейда** — так сохраняется связь между грейдом и ставкой.

**`person_emp_length` (стаж работы).** Прямой сильной связи с одним конкретным столбцом нет, поэтому пропуски заполним общей медианой по всему столбцу — она устойчива к выбросам (в отличие от среднего).

In [ ]:
rate_by_grade = df.groupby('loan_grade', observed=True)['loan_int_rate'].median()
display(rate_by_grade)

In [ ]:
df['loan_int_rate'] = df.groupby('loan_grade', observed=True)['loan_int_rate'].transform(
    lambda x: x.fillna(x.median())
)

emp_median = df['person_emp_length'].median()
age_median = df['person_age'].median()
df['person_emp_length'] = df['person_emp_length'].fillna(emp_median)
df['person_age'] = df['person_age'].fillna(age_median)

print('Итог по всему датасету:')
df.isnull().sum()

## 5. Анализ и обработка выбросов

В контексте машинного обучения и статистики **выброс (outlier)** — это объект (наблюдение, точка данных), который значительно отклоняется от остальных наблюдений в выборке.

**Типы выбросов по размерности пространства**

| Тип | Описание |
| :--- | :--- |
| **Одномерные (Univariate)** | Аномалия заметна при рассмотрении одного признака (например, возраст = 100 лет). |
| **Многомерные (Multivariate)** | Аномалия не видна ни по одному отдельному признаку, но проявляется при сочетании двух и более факторов. |

Проверим ключевые числовые столбцы по очереди.

In [ ]:
key_cols = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length'
]

df[key_cols].describe().T

### 5.1. Доход, сумма кредита, ставка, длина кредитной истории — метод межквартильного размаха (IQR)

Для остальных числовых признаков применим статистический метод поиска выбросов — **межквартильный размах (IQR)**:

1. Находим 1-й квартиль (Q1, 25% данных) и 3-й квартиль (Q3, 75% данных).
2. IQR = Q3 − Q1 — диапазон, в котором лежит "основная масса" данных.
3. Всё, что выходит за границы `[Q1 − 1.5·IQR; Q3 + 1.5·IQR]`, считается потенциальным выбросом.

Это стандартный статистический подход, не зависящий от предположений о форме распределения.

In [ ]:
def iqr_bounds(
    ser: pd.Series,
    lower_coef: float=1.5,
    upper_coef: float=1.5
) -> tuple[float, float]:
    q1 = ser.quantile(0.25)
    q3 = ser.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - lower_coef * iqr
    upper = q3 + upper_coef * iqr
    return lower, upper


def make_iqr_report(input_ser: pd.Series, **iqr_kwargs) -> dict[str, Any]:
    low, up = iqr_bounds(input_ser, **iqr_kwargs)
    n_out = ((input_ser < low) | (input_ser > up)).sum()
    return {
        'столбец': col,
        'нижняя_граница': round(low, 2),
        'верхняя_граница': round(up, 2),
        'кол-во_выбросов': n_out,
        'доля_выбросов_%': round(n_out / len(input_ser) * 100, 2)
    }
    

In [ ]:
view_cols = [
    'person_income', 'loan_amnt', 'loan_int_rate', 
    'loan_percent_income', 'cb_person_cred_hist_length'
]

report_rows = []
for col in view_cols:
    report_rows.append(make_iqr_report(df[col]))

pd.DataFrame(report_rows)

**Интерпретация:**

- `person_income` (доход) — по IQR довольно много "выбросов" сверху, но это ожидаемо: доход в реальном мире имеет длинный правый хвост (немного очень высокооплачиваемых людей). Такие значения — не ошибка, а нормальная асимметрия распределения. Удалять их массово нельзя — потеряем важную информацию. Уберём только экстремальные значения, остальное оставим как есть.
- `loan_amnt`, `loan_int_rate`, `loan_percent_income`, `cb_person_cred_hist_length` — значения физически правдоподобны (высокая ставка, большая сумма кредита — это часть нормальной вариативности кредитного рынка). Формально удалять их по IQR не будем, чтобы не терять данные о реальных высокорисковых заёмщиках (это как раз важно для модели кредитного риска).

Таким образом, для дохода применим точечную обрезку экстремальных значений, а IQR используем как **диагностический** инструмент, а не как автоматический фильтр — резать всё подряд по формуле для финансовых данных с естественной асимметрией нежелательно.

In [ ]:
INCOME_LIMIT = 5_000_000

print(f'Записей с доходом > {INCOME_LIMIT}:', (df['person_income'] > INCOME_LIMIT).sum())
df.loc[df['person_income'] > INCOME_LIMIT, ['person_income', 'loan_amnt', 'loan_status']]

In [ ]:

rows_before = len(df)
df_clean = df[df['person_income'] <= INCOME_LIMIT].reset_index(drop=True)
print(f'Удалено строк по экстремальному доходу: {rows_before - len(df_clean)}', end='\n\n')
print(f'Итоговый размер датасета после очистки выбросов: {df_clean.shape}')

## 6. Описательная статистика

Теперь, когда данные очищены, посмотрим на итоговую описательную статистику.


In [ ]:
df_clean.describe().T

In [ ]:
df_clean.describe(include=['category', 'bool']).T

In [ ]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
stats_extra = pd.DataFrame({
    'асимметрия (skew)': df_clean[numeric_cols].skew(),
    'эксцесс (kurtosis)': df_clean[numeric_cols].kurtosis()
}).round(2)
stats_extra

In [ ]:
df_clean.groupby('loan_grade', observed=True)[
    ['loan_int_rate', 'loan_amnt', 'loan_status']
].mean().round(2)

## 7. Профилирование данных

Профилирование — это сводный «паспорт» каждого столбца датасета: тип данных, число уникальных значений, базовые статистики. Соберём такой отчёт вручную средствами pandas/numpy.

In [ ]:
def profile_dataset(data):
    rows = []
    for col in data.columns:
        s = data[col]
        row = {
            'столбец': col,
            'тип': str(s.dtype),
            'непустых': s.notnull().sum(),
            'пропусков': s.isnull().sum(),
            'уникальных': s.nunique(),
            'дубликаты_значений_%': round((1 - s.nunique() / s.notnull().sum()) * 100, 2) if s.notnull().sum() > 0 else np.nan,
        }
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            row.update({
                'min': round(s.min(), 2),
                'среднее': round(s.mean(), 2),
                'медиана': round(s.median(), 2),
                'max': round(s.max(), 2),
                'станд. откл.': round(s.std(), 2),
            })
        else:
            top_value = s.mode().iloc[0] if not s.mode().empty else None
            row.update({
                'min': None, 'среднее': None, 
                'медиана': None, 'max': None, 
                'станд. откл.': None,
            })
            row['частое_значение'] = top_value
        rows.append(row)
    return pd.DataFrame(rows).set_index('столбец')


profile = profile_dataset(df_clean)
profile